# Pruebas de calidad: generar Bronze defectuoso

Crea `brz_dataco.pruebas_calidad.raw_supply_chain_defectuosa` a partir de **filas reales** de Bronze, con un defecto por caso y claves nuevas (`order_id` desde 9,000,001 y `order_item_id` desde 90,000,001) para que ninguna fila choque con datos reales ni con otra del mismo caso.

Es determinista: mismas filas base y mismos defectos en cada corrida.

La cuarentena esperada **no se escribe a mano**: se calcula desde las filas generadas y queda en `casos_esperados`. Los casos que solo advierten dejan su esperado en `casos_esperados_dq`.

| Caso | Filas | Defecto | Cuarentena esperada |
|---|---|---|---|
| CONTROL_SANO | 1,000 | ninguno | ninguna |
| ORDER_ID_INVALIDO | 1,000 | `order_id` vacío, `'0'` o NULL | todas las filas del caso, más las de prioridad que ganan |
| FECHA_PEDIDO_INVALIDA | 1,000 | fecha no parseable | todas las filas del caso, más las de prioridad que ganan |
| CLIENTE_NO_INFORMADO | 1,000 | `order_customer_id` vacío, `'0'` o NULL | todas las filas del caso, más las de prioridad que ganan |
| RIESGO_RETRASO_INVALIDO | 1,000 | `late_delivery_risk` distinto de 0 y 1 (2, -1 o vacío) | todas las filas del caso |
| CLIENTE_ID_INVALIDO | 1,000 | `customer_id` vacío, `'0'` o NULL | una fila por clave distinta |
| PRODUCTO_ID_INVALIDO | 1,000 | `product_card_id` vacío, `'0'` o NULL | una fila por clave distinta |
| UBICACION_INCOMPLETA | 1,000 | market, región o país vacío (zip único) | todas las filas del caso |
| ITEM_ID_INVALIDO | 1,000 | `order_item_id` vacío, `'0'` o NULL | una fila por clave distinta |
| CANTIDAD_INVALIDA | 1,000 | cantidad 0 o negativa | todas las filas del caso |
| PRIORIDAD_FECHA_Y_CLIENTE | 100 | dos defectos; gana FECHA_PEDIDO_INVALIDA | suma a FECHA |
| PRIORIDAD_CLIENTE_Y_RIESGO | 100 | dos defectos; gana CLIENTE_NO_INFORMADO | suma a CLIENTE |
| PRIORIDAD_ORDER_ID_Y_FECHA | 100 | dos defectos; gana ORDER_ID_INVALIDO | suma a ORDER_ID |
| FECHA_ENVIO_VACIA | 100 | `shipping_date_dateorders` vacía | ninguna (DQ-007 advierte) |
| ESTADO_PEDIDO_DESCONOCIDO | 100 | `order_status` fuera del catálogo | ninguna (DQ-010 advierte) |

In [0]:
from pyspark.sql import functions as F, Window

BRONZE_REAL = "brz_dataco.sharepoint.raw_supply_chain"
CATALOGO_PRUEBA = "brz_dataco"
ESQUEMA_PRUEBA = "pruebas_calidad"
BRONZE_PRUEBA = f"{CATALOGO_PRUEBA}.{ESQUEMA_PRUEBA}.raw_supply_chain_defectuosa"
TABLA_CASOS = f"{CATALOGO_PRUEBA}.{ESQUEMA_PRUEBA}.casos_prueba"
TABLA_ESPERADOS = f"{CATALOGO_PRUEBA}.{ESQUEMA_PRUEBA}.casos_esperados"

BASE_ORDER_ID = 9_000_000
BASE_ORDER_ITEM_ID = 90_000_000

N = 1000   # filas por caso principal
M = 100    # filas por caso de prioridad

CASOS = [
    ("CONTROL_SANO", N),
    ("ORDER_ID_INVALIDO", N),
    ("FECHA_PEDIDO_INVALIDA", N),
    ("CLIENTE_NO_INFORMADO", N),
    ("RIESGO_RETRASO_INVALIDO", N),
    ("CLIENTE_ID_INVALIDO", N),
    ("PRODUCTO_ID_INVALIDO", N),
    ("UBICACION_INCOMPLETA", N),
    ("ITEM_ID_INVALIDO", N),
    ("CANTIDAD_INVALIDA", N),
    ("PRIORIDAD_FECHA_Y_CLIENTE", M),
    ("PRIORIDAD_CLIENTE_Y_RIESGO", M),
    ("PRIORIDAD_ORDER_ID_Y_FECHA", M),
    ("FECHA_ENVIO_VACIA", M),
    ("ESTADO_PEDIDO_DESCONOCIDO", M),
]

rangos = []
inicio = 1
for nombre, filas in CASOS:
    rangos.append((nombre, inicio, inicio + filas - 1, filas))
    inicio += filas
TOTAL_FILAS = inicio - 1
print(f"Filas de prueba a generar: {TOTAL_FILAS}")

In [0]:
# Filas base reales: orden determinista por hash de la clave física.
orden_estable = Window.orderBy(F.xxhash64("order_id", "order_item_id"))

df_base = (
    spark.table(BRONZE_REAL)
    .withColumn("n", F.row_number().over(orden_estable))
    .filter(F.col("n") <= TOTAL_FILAS)
)

if df_base.count() < TOTAL_FILAS:
    raise ValueError("Bronze real no tiene filas suficientes para generar la prueba")

caso = F.lit(None).cast("string")
for nombre, ini, fin, _ in rangos:
    caso = F.when(F.col("n").between(ini, fin), F.lit(nombre)).otherwise(caso)

def variante(valores):
    # Reparte los valores defectuosos por fila (n % cantidad de variantes).
    arreglo = F.array(*[F.lit(v).cast("string") for v in valores])
    return F.element_at(arreglo, (F.col("n") % len(valores)).cast("int") + 1)

def en(*casos):
    return F.col("caso_prueba").isin(*casos)

def reemplazar(columna, casos, valores):
    return F.when(en(*casos), variante(valores)).otherwise(F.col(columna))

df_prueba = (
    df_base
    .withColumn("caso_prueba", caso)
    # Claves nuevas y únicas para todas las filas
    .withColumn("order_id", (F.col("n") + BASE_ORDER_ID).cast("string"))
    .withColumn("order_item_id", (F.col("n") + BASE_ORDER_ITEM_ID).cast("string"))
    # Defectos, de a una columna por regla
    .withColumn("order_id", reemplazar("order_id",
        ["ORDER_ID_INVALIDO", "PRIORIDAD_ORDER_ID_Y_FECHA"], ["", "0", None]))
    .withColumn("order_item_id", reemplazar("order_item_id",
        ["ITEM_ID_INVALIDO"], ["0", "", None]))
    .withColumn("order_date_dateorders", reemplazar("order_date_dateorders",
        ["FECHA_PEDIDO_INVALIDA", "PRIORIDAD_FECHA_Y_CLIENTE", "PRIORIDAD_ORDER_ID_Y_FECHA"],
        ["abc", "", "31/31/2018 10:00"]))
    .withColumn("order_customer_id", reemplazar("order_customer_id",
        ["CLIENTE_NO_INFORMADO", "PRIORIDAD_FECHA_Y_CLIENTE", "PRIORIDAD_CLIENTE_Y_RIESGO"],
        ["0", "", None]))
    .withColumn("late_delivery_risk", reemplazar("late_delivery_risk",
        ["RIESGO_RETRASO_INVALIDO", "PRIORIDAD_CLIENTE_Y_RIESGO"], ["2", "-1", ""]))
    .withColumn("customer_id", reemplazar("customer_id",
        ["CLIENTE_ID_INVALIDO"], ["0", "", None]))
    .withColumn("product_card_id", reemplazar("product_card_id",
        ["PRODUCTO_ID_INVALIDO"], ["0", "", None]))
    .withColumn("order_item_quantity", reemplazar("order_item_quantity",
        ["CANTIDAD_INVALIDA"], ["0", "-1", "-5"]))
    # Defectos que solo advierten (no mandan filas a cuarentena)
    .withColumn("shipping_date_dateorders", reemplazar("shipping_date_dateorders",
        ["FECHA_ENVIO_VACIA"], [""]))
    .withColumn("order_status", reemplazar("order_status",
        ["ESTADO_PEDIDO_DESCONOCIDO"], ["ESTADO_NUEVO"]))
    # Ubicación incompleta: un campo vacío por fila y zip único (una ubicación distinta por fila)
    .withColumn("market", F.when(en("UBICACION_INCOMPLETA") & (F.col("n") % 3 == 0), F.lit("")).otherwise(F.col("market")))
    .withColumn("order_region", F.when(en("UBICACION_INCOMPLETA") & (F.col("n") % 3 == 1), F.lit("")).otherwise(F.col("order_region")))
    .withColumn("order_country", F.when(en("UBICACION_INCOMPLETA") & (F.col("n") % 3 == 2), F.lit("")).otherwise(F.col("order_country")))
    .withColumn("order_zipcode", F.when(en("UBICACION_INCOMPLETA"), F.concat(F.lit("TEST-"), F.col("n").cast("string"))).otherwise(F.col("order_zipcode")))
    .drop("n")
)

(
    df_prueba.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true").saveAsTable(BRONZE_PRUEBA)
)
print(f"✓ {spark.table(BRONZE_PRUEBA).count()} filas en {BRONZE_PRUEBA}")
display(spark.table(BRONZE_PRUEBA).groupBy("caso_prueba").count().orderBy("caso_prueba"))

In [0]:
# Rangos de cada caso y cuarentena esperada por tabla Silver y motivo.
# Los rangos permiten ubicar cada caso en las tablas Silver por sus claves nuevas.
df_casos = spark.createDataFrame(
    [(nombre, ini, fin, filas,
      BASE_ORDER_ID + ini, BASE_ORDER_ID + fin,
      BASE_ORDER_ITEM_ID + ini, BASE_ORDER_ITEM_ID + fin)
     for nombre, ini, fin, filas in rangos],
    "caso_prueba STRING, n_inicio INT, n_fin INT, filas INT, "
    "order_id_inicio INT, order_id_fin INT, order_item_id_inicio INT, order_item_id_fin INT"
)
df_casos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_CASOS)

# Esperados calculados desde las filas generadas (df_prueba), no escritos a mano.
def filas_del_caso(*casos):
    return df_prueba.filter(F.col("caso_prueba").isin(*casos)).count()


def claves_distintas(columna, caso):
    # Silver agrupa por clave: vacío y NULL caen en NULL, '0' queda en 0.
    # Cada clave distinta deja una sola fila en cuarentena.
    clave = F.expr(f"try_cast(NULLIF(TRIM(CAST(`{columna}` AS STRING)), '') AS BIGINT)")
    return (
        df_prueba.filter(F.col("caso_prueba") == caso)
        .select(F.coalesce(clave.cast("string"), F.lit("NULL")).alias("clave"))
        .distinct().count()
    )


# Los casos de prioridad suman al motivo que gana.
ESPERADOS = [
    ("hd_pedidos", "ORDER_ID_INVALIDO", filas_del_caso("ORDER_ID_INVALIDO", "PRIORIDAD_ORDER_ID_Y_FECHA")),
    ("hd_pedidos", "FECHA_PEDIDO_INVALIDA", filas_del_caso("FECHA_PEDIDO_INVALIDA", "PRIORIDAD_FECHA_Y_CLIENTE")),
    ("hd_pedidos", "CLIENTE_NO_INFORMADO", filas_del_caso("CLIENTE_NO_INFORMADO", "PRIORIDAD_CLIENTE_Y_RIESGO")),
    ("hd_pedidos", "RIESGO_RETRASO_INVALIDO", filas_del_caso("RIESGO_RETRASO_INVALIDO")),
    ("md_clientes", "CLIENTE_ID_INVALIDO", claves_distintas("customer_id", "CLIENTE_ID_INVALIDO")),
    ("md_productos", "PRODUCTO_ID_INVALIDO", claves_distintas("product_card_id", "PRODUCTO_ID_INVALIDO")),
    ("md_ubicaciones", "UBICACION_INCOMPLETA", filas_del_caso("UBICACION_INCOMPLETA")),
    ("hd_pedido_detalle", "ITEM_ID_INVALIDO", claves_distintas("order_item_id", "ITEM_ID_INVALIDO")),
    ("hd_pedido_detalle", "CANTIDAD_INVALIDA", filas_del_caso("CANTIDAD_INVALIDA")),
]
spark.createDataFrame(ESPERADOS, "tabla_silver STRING, motivo_cuarentena STRING, filas_esperadas INT") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_ESPERADOS)

# Esperados de las reglas que solo advierten: filas que incumplen, calculadas desde df_prueba.
ESPERADOS_DQ = [
    ("DQ-007", filas_del_caso("FECHA_ENVIO_VACIA")),
    ("DQ-010", filas_del_caso("ESTADO_PEDIDO_DESCONOCIDO")),
]
TABLA_ESPERADOS_DQ = f"{CATALOGO_PRUEBA}.{ESQUEMA_PRUEBA}.casos_esperados_dq"
spark.createDataFrame(ESPERADOS_DQ, "regla_id STRING, filas_incumplen_esperadas INT") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLA_ESPERADOS_DQ)

print(f"✓ {TABLA_CASOS}")
print(f"✓ {TABLA_ESPERADOS}")
print(f"✓ {TABLA_ESPERADOS_DQ}")
display(spark.table(TABLA_ESPERADOS))
display(spark.table(TABLA_ESPERADOS_DQ))